In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score
from sklearn import metrics
from sklearn.linear_model import LinearRegression

# 1. EDA

In [ ]:
df = pd.read_csv("1 - Project Data.csv")

In [ ]:
df.head()

In [ ]:
df.isna().sum()

In [ ]:
df['Churn Value'].value_counts(normalize=True)

In [ ]:
df.info()

In [ ]:
df.dtypes

In [ ]:
df.shape

# 2. Cleaning

In [ ]:
df_clean = df.copy()

In [ ]:
df_clean['Total Charges'] = pd.to_numeric(df_clean['Total Charges'], errors='coerce').fillna(0.0)

In [ ]:
addons = ['Online Security', 'Online Backup', 'Device Protection',
          'Tech Support', 'Streaming TV', 'Streaming Movies']
df_clean[addons] = df_clean[addons].replace('No internet service', 'No')
df_clean['Multiple Lines'] = df_clean['Multiple Lines'].replace('No phone service', 'No')

# 3. Feature Engineering

In [ ]:
DROPS = [
    "Count", "Country", "State",
    "CustomerID",
    "Churn Label", "Churn Value",
    "Churn Reason",
    "Lat Long",
    "City", "Zip Code", "Latitude", "Longitude",
    "Total Charges",
]

customer_ids = df_clean["CustomerID"]
y = df_clean["Churn Value"]
X = df_clean.drop(columns=DROPS)

In [ ]:
X_enc = pd.get_dummies(X, drop_first=True, dtype=int)
FEATURE_COLUMNS = list(X_enc.columns)

In [ ]:
X_enc.head()

# 4. Scaling

In [ ]:
NUMERIC = ["Tenure Months", "Monthly Charges"]

scaler = StandardScaler()
X_enc[NUMERIC] = scaler.fit_transform(X_enc[NUMERIC])

In [ ]:
X_enc[NUMERIC].describe()

# 5. Fitting the Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_enc, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(max_iter=300, random_state=42)
model.fit(X_train, y_train)

In [ ]:
model.score(X_test, y_test)

In [ ]:
results = X_test.copy()
results[['prob_stay', 'prob_churn']] = model.predict_proba(X_test)
results['y_pred'] = np.where(results['prob_churn'] > .5, 1, 0)

roc_auc_score(y_test, results['prob_churn'])

In [ ]:
results.head()

# 6. Metrics

In [ ]:
def produce_confusion(positive_label, negative_label, cut_off, df, y_pred_name, y_real_name):
    pred = df[y_pred_name] if cut_off == 'binary' else np.where(df[y_pred_name] > cut_off, 1, 0)

    cm = confusion_matrix(df[y_real_name], pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, ax=ax, fmt='g', cmap='Blues', cbar=False)

    ax.set_xlabel('Predicted labels')
    ax.set_ylabel('Real labels')
    ax.set_title('Confusion Matrix')
    ax.xaxis.set_ticklabels([negative_label, positive_label])
    ax.yaxis.set_ticklabels([negative_label, positive_label])
    plt.show()

    acc = accuracy_score(df[y_real_name], pred)
    print('Test accuracy = ', acc)
    return acc

In [ ]:
results['y_real'] = y_test
produce_confusion('Churned', 'Retained', 'binary', results, 'y_pred', 'y_real')

In [ ]:
print(metrics.classification_report(y_test, results['y_pred']))

# 7. Finding the Best Threshold Value

In [ ]:
for cut in [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]:
    pred = np.where(results['prob_churn'] > cut, 1, 0)
    print(f"{cut:.2f}  acc {metrics.accuracy_score(y_test, pred):.3f}"
          f"  recall {metrics.recall_score(y_test, pred):.3f}"
          f"  precision {metrics.precision_score(y_test, pred):.3f}"
          f"  f1 {metrics.f1_score(y_test, pred):.3f}")

In [ ]:
results['y_pred'] = np.where(results['prob_churn'] > 0.35, 1, 0)
results.head()

# 8. The 500 Customers Most Likely to Churn

In [ ]:
scored = df_clean[["CustomerID", "Churn Value", "City", "Zip Code",
                   "Tenure Months", "Contract", "Internet Service",
                   "Online Security", "Tech Support", "Payment Method",
                   "Monthly Charges", "Total Charges"]].copy()

scored[["prob_stay", "prob_churn"]] = model.predict_proba(X_enc)

retained = scored[scored["Churn Value"] == 0].drop(columns="Churn Value")
mailer_list = retained.sort_values("prob_churn", ascending=False).head(500)

In [ ]:
mailer_list.head()

In [ ]:
mailer_list.to_csv("mailer_list_500.csv", index=False)

In [ ]:
ranked = results.sort_values("prob_churn", ascending=False)
n = len(ranked)

pct_contacted = np.arange(1, n + 1) / n * 100
pct_captured = ranked["y_real"].cumsum().values / ranked["y_real"].sum() * 100

mark = int(round(500 / len(retained) * n))

fig, ax = plt.subplots(figsize=(8.5, 6))
ax.plot(pct_contacted, pct_captured, color="#163E67", linewidth=2.2, label="Model ranking")
ax.plot([0, 100], [0, 100], color="#B9B9B9", linestyle="--", linewidth=1.4, label="Random targeting")

ax.plot(pct_contacted[mark-1], pct_captured[mark-1], "o", color="#C1121F", markersize=8, zorder=5)
ax.annotate(f"Top 500 equivalent\n{pct_contacted[mark-1]:.1f}% contacted, {pct_captured[mark-1]:.1f}% of churners",
            xy=(pct_contacted[mark-1], pct_captured[mark-1]), xytext=(28, 34),
            fontsize=10, arrowprops=dict(arrowstyle="->", color="#C1121F"))

ax.set_xlabel("% of customers contacted (ranked by churn risk)")
ax.set_ylabel("% of churners reached")
ax.set_title("Targeting by model risk reaches churners far faster than random", fontsize=12.5)
ax.legend(frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

In [ ]:
mailer_list["expected_saved_revenue"] = (
    mailer_list["prob_churn"] * mailer_list["Monthly Charges"] * 0.20)

print("Monthly revenue on the list :", mailer_list["Monthly Charges"].sum().round(2))
print("Expected churners           :", mailer_list["prob_churn"].sum().round(1))
print("Expected saves at 20%       :", (mailer_list["prob_churn"].sum() * 0.20).round(1))
print("Monthly revenue protected   :", mailer_list["expected_saved_revenue"].sum().round(2))
print("Annualised                  :", (mailer_list["expected_saved_revenue"].sum() * 12).round(2))

# 9. Churn Risk of Remaining Customers

In [ ]:
risk_register = retained.copy()
risk_register["churn_risk_pct"] = (risk_register["prob_churn"] * 100).round(1)
risk_register["risk_band"] = pd.cut(
    risk_register["prob_churn"],
    bins=[0, .10, .25, .45, .65, 1.0],
    labels=["Very low", "Low", "Medium", "High", "Very high"])

risk_register = risk_register.sort_values("CustomerID")
risk_register.to_csv("churn_risk_register.csv", index=False)

In [ ]:
band_summary = risk_register.groupby("risk_band", observed=True).agg(
    customers=("CustomerID", "size"),
    mean_risk=("churn_risk_pct", "mean"),
    monthly_revenue=("Monthly Charges", "sum"),
    mean_monthly_revenue=("Monthly Charges", "mean"),
)
band_summary["share"] = (band_summary["customers"] / len(risk_register) * 100).round(1)
band_summary = band_summary[["customers", "share", "mean_risk", "mean_monthly_revenue", "monthly_revenue"]].round(1)
band_summary

In [ ]:
band_by_service = pd.crosstab(
    risk_register["risk_band"],
    risk_register["Internet Service"],
    normalize="index").mul(100).round(1)
band_by_service

In [ ]:
cols = ["Fiber optic", "DSL", "No"]
names = ["Fibre optic", "DSL", "No internet"]
shades = ["#163E67", "#6A96C2", "#C9DCEF"]

fig, ax = plt.subplots(figsize=(9, 5.5))
bottom = np.zeros(len(band_by_service))

for c, nm, sh in zip(cols, names, shades):
    vals = band_by_service[c].values
    bars = ax.bar(band_by_service.index.astype(str), vals, bottom=bottom,
                  label=nm, color=sh, width=.65)
    for b, v in zip(bars, vals):
        if v >= 6:
            ax.text(b.get_x() + b.get_width()/2, b.get_y() + v/2, f"{v:.0f}%",
                    ha="center", va="center", fontsize=9,
                    color="white" if sh == "#163E67" else "#1A1A1A")
    bottom += vals

ax.set_ylabel("% of customers in band")
ax.set_xlabel("Churn risk band")
ax.set_ylim(0, 100)
ax.set_title("Churn risk is concentrated in fibre customers", fontsize=13)
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(.5, -.13))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

# 10. What factors most influence someone churning?

In [ ]:
coefficients = pd.DataFrame({
    "feature": X_enc.columns,
    "coefficient": model.coef_[0],
    "odds_ratio": np.exp(model.coef_[0]),
}).sort_values("coefficient", ascending=False).reset_index(drop=True)

coefficients["pct_change"] = ((coefficients["odds_ratio"] - 1) * 100).round(1)
coefficients

In [ ]:
plot_df = coefficients.sort_values("coefficient")
colours = np.where(plot_df["coefficient"] > 0, "#163E67", "#8FB4D9")

ax = plot_df.plot.barh(x="feature", y="coefficient", figsize=(9, 8),
                       legend=False, color=colours)
ax.axvline(0, color="black", linewidth=.8)

for bar, ratio in zip(ax.patches, plot_df["odds_ratio"]):
    w = bar.get_width()
    offset = 0.04 if w > 0 else -0.04
    ax.text(w + offset, bar.get_y() + bar.get_height()/2, f"{ratio:.2f}x",
            va="center", ha="left" if w > 0 else "right", fontsize=9)

ax.set_xlim(-2.1, 1.35)
ax.set_xlabel("Log-odds coefficient")
ax.set_ylabel("")
ax.set_title("What drives churn at Swan Teleco")
plt.tight_layout()
plt.show()

In [ ]:
service_cols = ['Online Security', 'Online Backup', 'Device Protection',
                'Tech Support', 'Streaming TV', 'Streaming Movies']

coefs = pd.Series(model.coef_[0], index=X_enc.columns)
retained_raw = df_clean[df_clean["Churn Value"] == 0]

incentive = pd.DataFrame({
    "odds_ratio": [np.exp(coefs[c + "_Yes"]) for c in service_cols],
    "churn_with": [df_clean[df_clean[c] == "Yes"]["Churn Value"].mean() * 100 for c in service_cols],
    "churn_without": [df_clean[df_clean[c] == "No"]["Churn Value"].mean() * 100 for c in service_cols],
    "headroom": [((retained_raw[c] == "No") & (retained_raw["Internet Service"] != "No")).sum() for c in service_cols],
}, index=service_cols)

incentive.sort_values("odds_ratio").round(2)

In [ ]:
plot_inc = incentive.sort_values("odds_ratio", ascending=False)
colours = np.where(plot_inc["odds_ratio"] > 1, "#C1121F", "#163E67")

fig, ax = plt.subplots(figsize=(9.5, 5))
bars = ax.barh(plot_inc.index, plot_inc["odds_ratio"], color=colours, height=.62, zorder=2)
ax.axvline(1, color="black", linewidth=1, zorder=3)

for b, orr, hd in zip(bars, plot_inc["odds_ratio"], plot_inc["headroom"]):
    x = max(orr + .03, 1.05)
    ax.text(x, b.get_y() + b.get_height()/2,
            f"{orr:.2f}x   ({hd:,} could still sign up)", va="center", fontsize=9, zorder=4)

ax.set_xlim(0, 2.35)
ax.set_xlabel("Churn odds ratio")
ax.set_title("Only security and support add-ons reduce churn", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

# 11. Overall Analysis

In [ ]:
churned = df_clean[df_clean["Churn Value"] == 1]

service_summary = df_clean.groupby("Internet Service").agg(
    customers=("CustomerID", "size"),
    churned=("Churn Value", "sum"),
    churn_rate=("Churn Value", "mean"),
)
service_summary["churn_rate"] = (service_summary["churn_rate"] * 100).round(1)
service_summary["share_of_base"] = (service_summary["customers"] / len(df_clean) * 100).round(1)

lost = churned.groupby("Internet Service")["Monthly Charges"].sum()
service_summary["monthly_lost"] = lost.round(2)
service_summary["share_of_lost"] = (lost / lost.sum() * 100).round(1)
service_summary["annualised_lost"] = (lost * 12).round(2)

service_summary

In [ ]:
order = ["Fiber optic", "DSL", "No"]
labels = {"Fiber optic": "Fibre optic", "DSL": "DSL", "No": "No internet"}
plot = service_summary.loc[order]

worst = plot["share_of_lost"].idxmax()
title = (f"{labels[worst]} is {plot.loc[worst, 'share_of_base']:.0f}% of customers "
         f"but {plot.loc[worst, 'share_of_lost']:.0f}% of lost revenue")

x = np.arange(len(order))
width = 0.38

fig, ax = plt.subplots(figsize=(9, 5.5))
b1 = ax.bar(x - width/2, plot["share_of_base"], width,
            label="Share of customer base", color="#8FB4D9")
b2 = ax.bar(x + width/2, plot["share_of_lost"], width,
            label="Share of lost revenue", color="#163E67")

for bars in (b1, b2):
    ax.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=10)

ax.set_xticks(x, [labels[o] for o in order])
ax.set_ylabel("Percentage")
ax.set_ylim(0, 95)
ax.set_title(title, fontsize=13)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
cols = ['Phone Service', 'Multiple Lines', 'Online Security', 'Online Backup',
        'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies']

P = pd.DataFrame({c: (df_clean[c] == 'Yes').astype(int) for c in cols})
P['DSL'] = (df_clean['Internet Service'] == 'DSL').astype(int)
P['Fibre'] = (df_clean['Internet Service'] == 'Fiber optic').astype(int)

lm = LinearRegression().fit(P, df_clean['Monthly Charges'])

prices = pd.Series(lm.coef_, index=P.columns).sort_values(ascending=False)
prices.map('${:,.2f}'.format)

In [ ]:
print(lm.score(P, df_clean['Monthly Charges']))
float(lm.intercept_)